# Holiday Package Prediction using. Multiple Models and implementating Random Forest

## Holiday Package Prediciton

### 1) Problem statement.
"Trips & Travel.Com" company wants to enable and establish a viable business model to expand the customer base.
One of the ways to expand the customer base is to introduce a new offering of packages. Currently, there are 5 types of packages the company is offering * Basic, Standard, Deluxe, Super Deluxe, King. Looking at the data of the last year, we observed that 18% of the customers purchased the packages. However, the marketing cost was quite high because customers were contacted at random without looking at the available information.
The company is now planning to launch a new product i.e. Wellness Tourism Package. Wellness Tourism is defined as Travel that allows the traveler to maintain, enhance or kick-start a healthy lifestyle, and support or increase one's sense of well-being.
However, this time company wants to harness the available data of existing and potential customers to make the marketing expenditure more efficient.
### 2) Data Collection.
The Dataset is collected from https://www.kaggle.com/datasets/susant4learning/holiday-package-purchase-prediction
The data consists of 20 column and 4888 rows.


In [55]:
# import all the necessary librariess
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold,cross_val_predict
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [56]:
# Load the dataset
dataset=pd.read_csv('dataset/Travel.csv')

In [5]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
777,200777,1,21.0,Self Enquiry,1,16.0,Salaried,Female,2,4.0,Basic,5.0,Single,2.0,0,3,1,0.0,Executive,16416.0
3407,203407,0,39.0,Self Enquiry,3,9.0,Salaried,Female,3,4.0,Deluxe,5.0,Married,5.0,0,4,0,2.0,Manager,25571.0
1536,201536,0,36.0,Company Invited,1,17.0,Salaried,Male,3,4.0,Deluxe,4.0,Unmarried,2.0,0,4,1,1.0,Manager,21499.0
3893,203893,0,33.0,Self Enquiry,1,9.0,Large Business,Male,4,4.0,Basic,5.0,Single,3.0,0,1,1,2.0,Executive,21117.0
2368,202368,0,43.0,Self Enquiry,1,9.0,Salaried,Male,3,5.0,King,3.0,Married,4.0,0,5,1,NaN,VP,34740.0


In [6]:
# Data Cleaning
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   object 
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   object 
 7   Gender                    4888 non-null   object 
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   object 
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   object 
 13  NumberOfTrips             4748 non-null   float64
 14  Passport

In [7]:
dataset.isna().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [8]:
dataset[dataset.isna().any(axis=1)]

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0
11,200011,0,NaN,Self Enquiry,1,21.0,Salaried,Female,2,4.0,Deluxe,3.0,Single,1.0,1,3,0,0.0,Manager,NaN
19,200019,0,NaN,Self Enquiry,1,8.0,Salaried,Male,2,3.0,Basic,3.0,Single,6.0,1,4,0,1.0,Executive,NaN
20,200020,0,NaN,Company Invited,1,17.0,Salaried,Female,3,2.0,Deluxe,3.0,Married,1.0,0,3,1,2.0,Manager,NaN
21,200021,1,NaN,Self Enquiry,3,15.0,Salaried,Male,2,4.0,Deluxe,5.0,Single,1.0,0,2,0,0.0,Manager,18407.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4850,204850,1,46.0,Self Enquiry,3,8.0,Salaried,Male,4,5.0,Deluxe,5.0,Married,NaN,0,4,1,3.0,Manager,36739.0
4851,204851,1,40.0,Self Enquiry,1,9.0,Salaried,Female,4,4.0,Basic,5.0,Married,NaN,1,1,1,1.0,Executive,35801.0
4868,204868,1,43.0,Company Invited,2,15.0,Salaried,Female,4,5.0,Basic,3.0,Married,NaN,0,5,1,2.0,Executive,36539.0
4869,204869,1,56.0,Self Enquiry,3,16.0,Small Business,Female,3,6.0,Basic,4.0,Single,NaN,0,1,1,2.0,Executive,37865.0


In [9]:
dataset.duplicated().sum()

np.int64(0)

In [10]:
# Split to categorical and numerical columns
categorical_cols=dataset.select_dtypes(include='O').columns
numerical_cols=dataset.select_dtypes(exclude='O').columns

In [11]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
1695,201695,0,59.0,Self Enquiry,1,9.0,Salaried,Male,3,4.0,Basic,3.0,Married,4.0,0,3,0,0.0,Executive,17177.0
3264,203264,0,36.0,Company Invited,3,18.0,Small Business,Male,3,4.0,Deluxe,3.0,Married,3.0,0,5,0,1.0,Manager,23646.0
2048,202048,0,38.0,Company Invited,1,8.0,Salaried,Female,3,1.0,Deluxe,5.0,Single,7.0,1,1,0,0.0,Manager,20980.0
2598,202598,0,33.0,Company Invited,3,15.0,Small Business,Fe Male,3,4.0,Standard,3.0,Unmarried,3.0,0,4,1,2.0,Senior Manager,27676.0
3383,203383,1,33.0,Self Enquiry,1,14.0,Salaried,Male,4,4.0,Deluxe,3.0,Divorced,3.0,0,3,1,2.0,Manager,23561.0


In [12]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [13]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [14]:
dataset['Gender']=dataset['Gender'].str.replace('Fe Male','Female')

In [15]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male      2916
Female    1972
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [16]:
dataset['MaritalStatus']=dataset['MaritalStatus'].str.replace('Single','Unmarried')

In [17]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male      2916
Female    1972
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Unmarried    1598
Divorced      950
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [18]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
336,200336,1,29.0,Self Enquiry,1,14.0,Salaried,Male,3,5.0,Basic,5.0,Divorced,2.0,1,3,1,1.0,Executive,17119.0
429,200429,0,46.0,Self Enquiry,1,8.0,Salaried,Male,2,3.0,Super Deluxe,3.0,Divorced,7.0,0,5,0,0.0,AVP,32861.0
2845,202845,0,57.0,Company Invited,3,13.0,Salaried,Female,3,3.0,Super Deluxe,5.0,Divorced,2.0,0,3,1,1.0,AVP,31890.0
2651,202651,0,37.0,Company Invited,1,25.0,Small Business,Female,4,4.0,Basic,3.0,Unmarried,3.0,0,3,1,3.0,Executive,20831.0
2114,202114,0,31.0,Self Enquiry,1,17.0,Salaried,Male,2,3.0,Basic,3.0,Married,4.0,1,3,0,0.0,Executive,17356.0


In [19]:
dataset['Age'] = dataset['Age'].fillna(dataset['Age'].median())
dataset['TypeofContact'] = dataset['TypeofContact'].fillna(dataset['TypeofContact'].mode()[0])
dataset['DurationOfPitch'] = dataset['DurationOfPitch'].fillna(dataset['DurationOfPitch'].median())
dataset['NumberOfFollowups'] = dataset['NumberOfFollowups'].fillna(dataset['NumberOfFollowups'].mode()[0])
dataset['PreferredPropertyStar'] = dataset['PreferredPropertyStar'].fillna(dataset['PreferredPropertyStar'].mode()[0])
dataset['NumberOfTrips'] = dataset['NumberOfTrips'].fillna(dataset['NumberOfTrips'].median())
dataset['NumberOfChildrenVisiting'] = dataset['NumberOfChildrenVisiting'].fillna(dataset['NumberOfChildrenVisiting'].mode()[0])
dataset['MonthlyIncome'] = dataset['MonthlyIncome'].fillna(dataset['MonthlyIncome'].median())


In [20]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
3640,203640,0,42.0,Self Enquiry,3,30.0,Salaried,Male,4,4.0,Standard,3.0,Unmarried,5.0,1,3,0,3.0,Senior Manager,25760.0
3804,203804,0,32.0,Self Enquiry,1,11.0,Small Business,Female,4,2.0,Basic,3.0,Married,2.0,0,5,1,3.0,Executive,22656.0
1322,201322,0,46.0,Self Enquiry,1,8.0,Salaried,Male,2,3.0,Standard,3.0,Married,4.0,0,1,1,1.0,Senior Manager,23578.0
445,200445,0,55.0,Self Enquiry,3,24.0,Salaried,Female,2,3.0,Super Deluxe,4.0,Unmarried,4.0,0,2,0,1.0,AVP,31835.0
1022,201022,0,36.0,Company Invited,1,11.0,Large Business,Male,2,1.0,Basic,3.0,Unmarried,1.0,0,5,1,0.0,Executive,18500.0


In [21]:
dataset.isna().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [22]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [23]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,3,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,1.0,Manager,26576.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,4,5.0,Basic,3.0,Unmarried,3.0,1,3,1,2.0,Executive,21212.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4,4.0,Standard,4.0,Married,7.0,0,1,1,3.0,Senior Manager,31820.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,3,4.0,Basic,3.0,Unmarried,3.0,0,5,0,2.0,Executive,20289.0


In [24]:
# Feature Engineerig

dataset['TotalNoOfPeople']=dataset['NumberOfPersonVisiting']+dataset['NumberOfChildrenVisiting']

In [25]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0,3.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0,5.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0,3.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0,3.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,3,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,1.0,Manager,26576.0,4.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,4,5.0,Basic,3.0,Unmarried,3.0,1,3,1,2.0,Executive,21212.0,6.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4,4.0,Standard,4.0,Married,7.0,0,1,1,3.0,Senior Manager,31820.0,7.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,3,4.0,Basic,3.0,Unmarried,3.0,0,5,0,2.0,Executive,20289.0,5.0


In [26]:
dataset.drop(['NumberOfChildrenVisiting','NumberOfPersonVisiting'],axis=1,inplace=True)

In [27]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [28]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              4888 non-null   int64  
 1   ProdTaken               4888 non-null   int64  
 2   Age                     4888 non-null   float64
 3   TypeofContact           4888 non-null   object 
 4   CityTier                4888 non-null   int64  
 5   DurationOfPitch         4888 non-null   float64
 6   Occupation              4888 non-null   object 
 7   Gender                  4888 non-null   object 
 8   NumberOfFollowups       4888 non-null   float64
 9   ProductPitched          4888 non-null   object 
 10  PreferredPropertyStar   4888 non-null   float64
 11  MaritalStatus           4888 non-null   object 
 12  NumberOfTrips           4888 non-null   float64
 13  Passport                4888 non-null   int64  
 14  PitchSatisfactionScore  4888 non-null   

In [29]:
# Extract the number of numerical cols and categorical cols
categorical_cols=dataset.select_dtypes(include='O').columns
numerical_cols=dataset.select_dtypes(exclude='O').columns

In [30]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [31]:
numerical_cols

Index(['CustomerID', 'ProdTaken', 'Age', 'CityTier', 'DurationOfPitch',
       'NumberOfFollowups', 'PreferredPropertyStar', 'NumberOfTrips',
       'Passport', 'PitchSatisfactionScore', 'OwnCar', 'MonthlyIncome',
       'TotalNoOfPeople'],
      dtype='object')

In [32]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [33]:
# Divide the Dataset
X=dataset.drop('ProdTaken',axis=1)
y=dataset['ProdTaken']

In [34]:
X

,CustomerID,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,200001,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,200002,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,200003,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,200004,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,204884,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,204885,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,204886,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [35]:
y

0       1
1       0
2       1
3       0
4       0
       ..
4883    1
4884    1
4885    1
4886    1
4887    1
Name: ProdTaken, Length: 4888, dtype: int64

In [36]:
# Train test Split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [37]:
X_train

,CustomerID,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
736,200736,48.0,Self Enquiry,1,10.0,Salaried,Male,4.0,Standard,3.0,Unmarried,1.0,0,5,1,Senior Manager,25999.0,4.0
1615,201615,30.0,Self Enquiry,1,11.0,Large Business,Female,3.0,Basic,5.0,Married,6.0,0,5,0,Executive,18204.0,3.0
336,200336,29.0,Self Enquiry,1,14.0,Salaried,Male,5.0,Basic,5.0,Divorced,2.0,1,3,1,Executive,17119.0,4.0
4526,204526,29.0,Self Enquiry,3,9.0,Small Business,Female,4.0,Deluxe,4.0,Married,3.0,1,3,1,Manager,23457.0,5.0
2665,202665,34.0,Self Enquiry,1,11.0,Small Business,Female,5.0,Basic,4.0,Divorced,8.0,0,4,0,Executive,21300.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,204426,28.0,Self Enquiry,1,10.0,Small Business,Male,5.0,Basic,3.0,Unmarried,2.0,0,1,1,Executive,20723.0,5.0
466,200466,41.0,Self Enquiry,3,8.0,Salaried,Female,3.0,Super Deluxe,5.0,Divorced,1.0,0,5,1,AVP,31595.0,4.0
3092,203092,38.0,Company Invited,3,28.0,Small Business,Female,4.0,Basic,3.0,Divorced,7.0,0,2,1,Executive,21651.0,5.0
3772,203772,28.0,Self Enquiry,3,30.0,Small Business,Female,5.0,Deluxe,3.0,Married,3.0,0,1,1,Manager,22218.0,5.0


In [38]:
y_test

144     0
79      0
2098    0
4738    0
2858    1
       ..
2570    1
3901    0
3364    0
3639    0
1962    0
Name: ProdTaken, Length: 1467, dtype: int64

In [39]:
# Extract the number of numerical cols and categorical cols
categorical_cols=X_train.select_dtypes(include='O').columns
numerical_cols=X_train.select_dtypes(exclude='O').columns

In [40]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [41]:

# transformer = ColumnTransformer(
#     transformers=[
#         ('oneHotEncoder', OneHotEncoder(drop='first'), categorical_cols)
#         ('Scaler',StandardScaler(),numerical_cols)
#     ],
#     remainder='passthrough'   # keeps non-categorical columns if any
# )

# # Fit and transform
# X_train_transformed = transformer.fit_transform(X_train)

# # Get column names
# encoded_cols = transformer.get_feature_names_out()

# # Convert to DataFrame with correct column names
# X_train = pd.DataFrame(
#     X_train_transformed,
#     columns=encoded_cols,
#     index=X_train.index
# )


In [42]:
# X_test_trasformed=transformer.transform(X_test)

In [43]:
X_train

,CustomerID,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
736,200736,48.0,Self Enquiry,1,10.0,Salaried,Male,4.0,Standard,3.0,Unmarried,1.0,0,5,1,Senior Manager,25999.0,4.0
1615,201615,30.0,Self Enquiry,1,11.0,Large Business,Female,3.0,Basic,5.0,Married,6.0,0,5,0,Executive,18204.0,3.0
336,200336,29.0,Self Enquiry,1,14.0,Salaried,Male,5.0,Basic,5.0,Divorced,2.0,1,3,1,Executive,17119.0,4.0
4526,204526,29.0,Self Enquiry,3,9.0,Small Business,Female,4.0,Deluxe,4.0,Married,3.0,1,3,1,Manager,23457.0,5.0
2665,202665,34.0,Self Enquiry,1,11.0,Small Business,Female,5.0,Basic,4.0,Divorced,8.0,0,4,0,Executive,21300.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,204426,28.0,Self Enquiry,1,10.0,Small Business,Male,5.0,Basic,3.0,Unmarried,2.0,0,1,1,Executive,20723.0,5.0
466,200466,41.0,Self Enquiry,3,8.0,Salaried,Female,3.0,Super Deluxe,5.0,Divorced,1.0,0,5,1,AVP,31595.0,4.0
3092,203092,38.0,Company Invited,3,28.0,Small Business,Female,4.0,Basic,3.0,Divorced,7.0,0,2,1,Executive,21651.0,5.0
3772,203772,28.0,Self Enquiry,3,30.0,Small Business,Female,5.0,Deluxe,3.0,Married,3.0,0,1,1,Manager,22218.0,5.0


In [44]:
# Now Both The features are Encoded Now We will Build the pipeline
transformer = ColumnTransformer(
    transformers=[
        ('oneHotEncoder', OneHotEncoder(drop='first',handle_unknown='ignore'), categorical_cols),
        ('Scaler',StandardScaler(),numerical_cols)
    ],
    remainder='passthrough'   # keeps non-categorical columns if any
)

def checkScores(models):
    cv=StratifiedKFold(shuffle=True,random_state=42)
    for m in models:
        pipe=Pipeline([
            ('preprocessing',transformer),
            ('model',m)
        ])
        print(f'----------------------{m} Model Trained-------------------')
        y_predicted=cross_val_predict(estimator=pipe,X=X_train,y=y_train,cv=cv)
        
        print(f'----->{m} Model Metricess On traing data')
        print(f'Accuracy Score: {accuracy_score(y_pred=y_predicted,y_true=y_train)}')
        print(f'Confusion Matrix:\n {confusion_matrix(y_pred=y_predicted,y_true=y_train)}')
        print(f'Classification report: \n {classification_report(y_pred=y_predicted,y_true=y_train)}')
        pipe.fit(X_train, y_train)

        y_predicted=pipe.predict(X_test)
        
        print(f'---------->{m} Model Metricess On traing data')
        print(f'Accuracy Score: {accuracy_score(y_pred=y_predicted,y_true=y_test)}')
        print(f'Confusion Matrix:\n {confusion_matrix(y_pred=y_predicted,y_true=y_test)}')
        print(f'Classification report: \n {classification_report(y_pred=y_predicted,y_true=y_test)}')

In [57]:

logistic=LogisticRegression(max_iter=10000)
decision=DecisionTreeClassifier()
forest=RandomForestClassifier()
knn=KNeighborsClassifier()
gradientBoost=GradientBoostingClassifier()
xgboost=XGBClassifier()
checkScores([logistic,decision,forest,knn,gradientBoost,xgboost])

----------------------LogisticRegression(max_iter=10000) Model Trained-------------------


d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


----->LogisticRegression(max_iter=10000) Model Metricess On traing data
Accuracy Score: 0.8436129786612102
Confusion Matrix:
 [[2689   86]
 [ 449  197]]
Classification report: 
               precision    recall  f1-score   support

           0       0.86      0.97      0.91      2775
           1       0.70      0.30      0.42       646

    accuracy                           0.84      3421
   macro avg       0.78      0.64      0.67      3421
weighted avg       0.83      0.84      0.82      3421

---------->LogisticRegression(max_iter=10000) Model Metricess On traing data
Accuracy Score: 0.8398091342876619
Confusion Matrix:
 [[1150   43]
 [ 192   82]]
Classification report: 
               precision    recall  f1-score   support

           0       0.86      0.96      0.91      1193
           1       0.66      0.30      0.41       274

    accuracy                           0.84      1467
   macro avg       0.76      0.63      0.66      1467
weighted avg       0.82      0.84      0

d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


----->DecisionTreeClassifier() Model Metricess On traing data
Accuracy Score: 0.8640748319204911
Confusion Matrix:
 [[2536  239]
 [ 226  420]]
Classification report: 
               precision    recall  f1-score   support

           0       0.92      0.91      0.92      2775
           1       0.64      0.65      0.64       646

    accuracy                           0.86      3421
   macro avg       0.78      0.78      0.78      3421
weighted avg       0.87      0.86      0.86      3421

---------->DecisionTreeClassifier() Model Metricess On traing data
Accuracy Score: 0.9038854805725971
Confusion Matrix:
 [[1124   69]
 [  72  202]]
Classification report: 
               precision    recall  f1-score   support

           0       0.94      0.94      0.94      1193
           1       0.75      0.74      0.74       274

    accuracy                           0.90      1467
   macro avg       0.84      0.84      0.84      1467
weighted avg       0.90      0.90      0.90      1467

-----

...

----->RandomForestClassifier() Model Metricess On traing data
Accuracy Score: 0.889505992399883
Confusion Matrix:
 [[2732   43]
 [ 335  311]]
Classification report: 
               precision    recall  f1-score   support

           0       0.89      0.98      0.94      2775
           1       0.88      0.48      0.62       646

    accuracy                           0.89      3421
   macro avg       0.88      0.73      0.78      3421
weighted avg       0.89      0.89      0.88      3421

---------->RandomForestClassifier() Model Metricess On traing data
Accuracy Score: 0.9079754601226994
Confusion Matrix:
 [[1179   14]
 [ 121  153]]
Classification report: 
               precision    recall  f1-score   support

           0       0.91      0.99      0.95      1193
           1       0.92      0.56      0.69       274

    accuracy                           0.91      1467
   macro avg       0.91      0.77      0.82      1467
weighted avg       0.91      0.91      0.90      1467

------

d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


----->GradientBoostingClassifier() Model Metricess On traing data
Accuracy Score: 0.869921075708857
Confusion Matrix:
 [[2705   70]
 [ 375  271]]
Classification report: 
               precision    recall  f1-score   support

           0       0.88      0.97      0.92      2775
           1       0.79      0.42      0.55       646

    accuracy                           0.87      3421
   macro avg       0.84      0.70      0.74      3421
weighted avg       0.86      0.87      0.85      3421

---------->GradientBoostingClassifier() Model Metricess On traing data
Accuracy Score: 0.8657123381049762
Confusion Matrix:
 [[1165   28]
 [ 169  105]]
Classification report: 
               precision    recall  f1-score   support

           0       0.87      0.98      0.92      1193
           1       0.79      0.38      0.52       274

    accuracy                           0.87      1467
   macro avg       0.83      0.68      0.72      1467
weighted avg       0.86      0.87      0.85      1467

d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


----->XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...) Model Metricess On traing data
Accuracy Score: 0.8982753580824321
Confusion Matrix:
 [[2694   81]
 [ 267  379]]
Classification report: 
               precision    recall  f1-score   support

           0       0.91      0.97      0.

In [52]:
def tune(models):
    
    for model, params in models.items():
        cv=StratifiedKFold(random_state=42,shuffle=True)
        pipe = Pipeline([
            ('preprocessing', transformer),
            ('model', model)
        ])
        
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=params,
            scoring='accuracy',
            cv=cv,
            n_jobs=-1
        )
        
        print(f'\n---------------------- {model.__class__.__name__} ----------------------')
        
        # Fit on TRAIN data only
        grid.fit(X_train, y_train)
        
        print("Best Parameters:")
        print(grid.best_params_)

        print("Best Estimators:")
        print(grid.best_estimator_)
        
        print("Best CV Accuracy:")
        print(grid.best_score_)
        
        # Test set evaluation
        y_test_pred = grid.predict(X_test)
        
        print("\n-----> Test Metrics")
        print("Accuracy:", accuracy_score(y_test, y_test_pred))
        print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
        print("Classification Report:\n", classification_report(y_test, y_test_pred))


In [58]:

logistic=LogisticRegression(max_iter=10000,
    class_weight='balanced')
decision=DecisionTreeClassifier()
forest=RandomForestClassifier()
knn=KNeighborsClassifier()
gradientBoost=GradientBoostingClassifier()
xgboost=XGBClassifier()

logistic_params = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['liblinear', 'lbfgs']
}

decision_params = {
    'model__max_depth': [None, 5, 10, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 5],
    'model__criterion': ['gini', 'entropy']
}

forest_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 'log2']
}

knn_params = {
    'model__n_neighbors': [3, 5, 7, 9],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan']
}
gradient_params={
    'model__loss':['log_loss', 'exponential'],
    'model__n_estimators': [100, 200,500,1000],
    'model__criterion':['friedman_mse', 'squared_error']

}

xg_paramList = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
    'model__gamma': [0, 1, 5]
}


models = {
    logistic: logistic_params,
    decision: decision_params,
    forest: forest_params,
    knn: knn_params,
    gradientBoost:gradient_params,
    xgboost:xg_paramList
}


In [59]:
tune(models=models)


---------------------- LogisticRegression ----------------------
Best Parameters:
{'model__C': 10, 'model__solver': 'liblinear'}
Best Estimators:
Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('oneHotEncoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')),
                                                 ('Scaler', StandardScaler(),
                                                  Index(['CustomerID', 'Age', 'CityTier', 'DurationOfPitch', 'NumberOfFollowups',
       'PreferredPropertyStar', 'NumberOfTrips', 'Passport',
       'PitchSatisfactionScore', 'OwnCar', 'MonthlyIncome', 'TotalNoOfPeopl

In [ ]:
# From this the Gradient Boost Give 89% accuracy with best parameters